# GF4 CUDA Kernels on Colab

Builds and runs the hand-written CUDA kernels in `CUDA_FP4_Test` (Hadamard / GF4 encode-decode / Hessian weight-quant / fused E2M1 GEMV) on a Colab GPU, plus profiling.

**What this is:** the real `.cu` kernels + `bench.py` (correctness + fused-vs-naive timing) and `llm_quant_eval.py` (full pipeline on a real model). The eval path is *fake-quant + fp16 GEMM* — it measures accuracy, not inference speed. The kernel timings in `bench.py` are the actual kernel-speed evidence.

**Runtime:** set **Runtime → Change runtime type → GPU** (T4/L4/A100/H100 all work). Bigger GPU only matters for `llm_quant_eval.py` on larger models.

## 0. GPU + toolchain check (and pin the compile arch)

In [ ]:
!nvidia-smi
!nvcc --version
import os, torch
cap = torch.cuda.get_device_capability()
arch = f"{cap[0]}.{cap[1]}"   # 7.5 T4 | 8.0 A100 | 8.9 L4 | 9.0 H100
os.environ["TORCH_CUDA_ARCH_LIST"] = arch    # pin so the JIT build compiles for THIS GPU only (faster, avoids arch warnings)
print("torch", torch.__version__, "| CUDA", torch.version.cuda,
      "| GPU", torch.cuda.get_device_name(0), "| sm_", arch.replace('.',''))

## 1. Get the code onto Colab

`CUDA_FP4_Test/` is untracked in git, so cloning won't include it. Upload the tarball instead.

**On your laptop**, the tarball is already built at:
`Python_Jenks_Test/Jenks_Tests/CUDA_FP4_Test/cuda_fp4_test.tar.gz`
(re-create anytime with: `cd CUDA_FP4_Test && tar czf cuda_fp4_test.tar.gz *.cu *.cuh *.cpp *.py README.md`)

Run the cell, then pick that file in the upload dialog.

In [ ]:
# OPTION A (recommended): upload cuda_fp4_test.tar.gz
from google.colab import files
up = files.upload()                      # <- choose cuda_fp4_test.tar.gz
!mkdir -p /content/CUDA_FP4_Test && tar xzf cuda_fp4_test.tar.gz -C /content/CUDA_FP4_Test
%cd /content/CUDA_FP4_Test
!ls

In [ ]:
# --- Alternatives (leave commented unless you need them) ---
# OPTION B: Google Drive (put the CUDA_FP4_Test folder in MyDrive first)
# from google.colab import drive; drive.mount('/content/drive')
# !cp -r "/content/drive/MyDrive/CUDA_FP4_Test" /content/
# %cd /content/CUDA_FP4_Test
#
# OPTION C: git clone (ONLY works after you commit+push CUDA_FP4_Test)
# !git clone https://github.com/jvap2/If_I_could_turn_back_time.git
# %cd If_I_could_turn_back_time/Python_Jenks_Test/Jenks_Tests/CUDA_FP4_Test

In [ ]:
# Build deps. torch is preinstalled on Colab; ninja speeds the JIT build.
!pip -q install ninja
# transformers/datasets/accelerate are only needed for llm_quant_eval.py (section 3):
!pip -q install "transformers==4.46.3" datasets accelerate

## 2. Build + run the kernels (`bench.py`)

First run JIT-compiles all `.cu` into the `gf4_kernels` extension (1–3 min), then runs every kernel's correctness check and the fused-vs-naive timing races. This is the CUDA work to examine — the timing lines are the real kernel-speed numbers.

The build is cached under `~/.cache/torch_extensions`, so re-runs are instant unless the runtime resets.

In [ ]:
!TORCH_CUDA_ARCH_LIST={arch} python3 bench.py

## 3. (optional) Full pipeline on a real model

Runs the W4A16 / W4A4 / W4A16-Hadamard ablations on a real LLM using the compiled kernels. `opt-125m` needs no GPU memory tricks. For bigger models on A100/H100 add `--device-map --max-memory "0=..GiB,cpu=..GiB"`.

**Note:** `llm_quant_eval.py` currently calls `login(token=...)` with a hardcoded HF token — fine for a private run, but if you share this notebook, replace it with `login(token=os.environ.get('HF_TOKEN'))` and set `HF_TOKEN` in Colab (🔑 Secrets).

In [ ]:
!TORCH_CUDA_ARCH_LIST={arch} python3 llm_quant_eval.py --model facebook/opt-125m --num-calib-windows 2 --num-eval-windows 5

## 4. Profiling

| Tool | What it gives | On Colab? |
|---|---|---|
| **Nsight Systems (`nsys`)** | Timeline trace: kernel launches, durations, memcpy, gaps, overlap | ✅ usually works (CUPTI tracing) |
| **Nsight Compute (`ncu`)** | Per-kernel metrics: occupancy, memory throughput, roofline | ❌ usually **blocked** (`ERR_NVGPUCTRPERM` — needs GPU perf-counter/admin perms Colab doesn't grant) |
| **`torch.profiler`** | Per-op/kernel CUDA times, Chrome trace | ✅ works, pure-Python |

**Bottom line:** do timeline profiling with `nsys` here; keep the detailed `ncu` kernel analysis (your `profile_hbm.py` / `ncu_hbm.csv`) on the desktop where you have counter permissions. `torch.profiler` is the Colab-native fallback for per-kernel times.

### 4a. Nsight Systems (timeline trace)

In [ ]:
# Check availability (nsys ships with the CUDA toolkit on most Colab images).
!which nsys && nsys --version || echo "nsys not on PATH — try:  apt-get -q update && apt-get -q install -y nsight-systems-cli"

In [ ]:
# Trace bench.py. --trace uses CUPTI (allowed on Colab); produces gf4_bench.nsys-rep.
!TORCH_CUDA_ARCH_LIST={arch} nsys profile \
    --trace=cuda,nvtx,osrt --force-overwrite=true \
    -o gf4_bench python3 bench.py
print("\n--- report summary ---")
!nsys stats --report gpukernsum gf4_bench.nsys-rep 2>/dev/null | head -40

In [ ]:
# Download the .nsys-rep and open it in the Nsight Systems GUI on your machine.
from google.colab import files
files.download("gf4_bench.nsys-rep")

### 4b. Nsight Compute (expected to fail on Colab — shown for the record)

In [ ]:
# Expect: ==ERROR== ERR_NVGPUCTRPERM ... GPU Performance Counters.
# This is a Colab permission limit, not a bug in the kernels. Run ncu on the desktop.
!which ncu && ncu --version || echo "ncu not present"
!ncu --set basic --launch-count 3 -f -o gf4_ncu \
    python3 -c "import torch; from torch.utils.cpp_extension import load" 2>&1 | head -15

### 4c. `torch.profiler` (Colab-native per-kernel timing)

Profiles a representative loop of the GF4 encode/decode + Hadamard kernels in-process. Prints the CUDA-time table and writes a Chrome trace (`chrome://tracing` or `edge://tracing` to view).

In [ ]:
import torch
from torch.utils.cpp_extension import load
from torch.profiler import profile, ProfilerActivity

ext = load(name="gf4_kernels", sources=[
    "bindings.cpp", "hadamard_kernel.cu", "gf4_encode_kernel.cu",
    "hessian_weight_quant_kernel.cu", "e2m1_fused_gemv_kernel.cu"],
    extra_cuda_cflags=["-O3", "--use_fast_math"], verbose=False)

# Representative activation tensor + a random-sign Hadamard vector.
HAD_BLOCK, GF4_BLOCK, CLIP = 32, 32, 2.5
x = torch.randn(8192, 4096, device="cuda").reshape(-1, HAD_BLOCK).contiguous()
d_sign = (torch.randint(0, 2, (HAD_BLOCK,), device="cuda") * 2 - 1).float()
mu = torch.zeros(HAD_BLOCK, device="cuda")

def step():
    xh = ext.hadamard_fwht(x, HAD_BLOCK, d_sign)
    nb = xh.numel() // GF4_BLOCK
    codes, scales = ext.gf4_encode(xh.reshape(-1).contiguous(), CLIP, True, mu)
    xq = ext.gf4_decode(codes, scales, nb)
    torch.cuda.synchronize()

for _ in range(3):
    step()   # warm up (JIT, allocator)

with profile(activities=[ProfilerActivity.CPU, ProfilerActivity.CUDA]) as prof:
    for _ in range(20):
        step()

print(prof.key_averages().table(sort_by="cuda_time_total", row_limit=15))
prof.export_chrome_trace("gf4_trace.json")
from google.colab import files; files.download("gf4_trace.json")